1. Import basic library

In [ ]:
import pandas as pd
import numpy as np
import time
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.preprocessing import LabelEncoder
import gower
import warnings
warnings.filterwarnings('ignore')

2. Completa code for Attack I

================================================================================
ATTACK MODEL 3: k-NN BASED ATTACK EVALUATION PIPELINE
================================================================================

PIPELINE OVERVIEW:
-----------------
This pipeline evaluates the effectiveness of different distance metrics for 
attribute inference attacks using k-Nearest Neighbors (k-NN).

Unlike Attack Model 2 (which used meta-features + XGBoost), this approach uses
a simpler, more interpretable k-NN classifier that leverages the synthetic
dataset (SD) as the reference knowledge base.

ATTACK SCENARIO (Same as Attack Model 2):
----------------------------------------
The attacker has access to:
1. SYNTHETIC DATASET (SD): 10,000 synthetic samples WITH labels (diagnosis known)
2. TRAINING RECORDS (TR): Not directly used as training data in this attack!
   The attacker uses SD as the reference dataset instead.
3. VICTIM RECORDS (VR): 2,000 real records where diagnosis is HIDDEN
4. BASELINE RECORDS (BR/M2): 2,000 real records NOT in training set

ATTACK MECHANISM:
----------------
For each victim record, the attacker:
1. Computes distances to ALL synthetic records (using various distance metrics)
2. Finds the k nearest synthetic neighbors
3. Performs weighted majority vote (inverse distance weighting)
4. Predicts the diagnosis class based on neighbors' labels

The key intuition: If synthetic data preserves the structure of the real data,
then a victim record should have neighbors with the correct diagnosis.

WHAT THIS PIPELINE EVALUATES:
----------------------------
This pipeline systematically compares DIFFERENT DISTANCE METRICS to determine
which ones work best for the attack:

METRICS TESTED (13 total):
-------------------------
CONTINUOUS/NUMERIC METRICS (11):
  - euclidean, l2: Standard Euclidean distance
  - sqeuclidean: Squared Euclidean
  - manhattan, l1, cityblock: Manhattan distance
  - cosine: Cosine similarity (angular distance)
  - braycurtis: Bray-Curtis dissimilarity
  - correlation: 1 - Pearson correlation
  - canberra: Canberra distance (weighted absolute difference)
  - nan_euclidean: Euclidean with NaN handling

BINARY METRICS (7, but used conditionally):
  - dice: Dice coefficient
  - jaccard: Jaccard similarity
  - sokalmichener, sokalsneath, yule, russellrao, rogerstanimoto

SPECIAL METRIC (1):
  - GOWER: Mixed-type distance (handles categorical + numeric data)

Additionally, the pipeline tests different k VALUES (39 values):
  Fine-grained: 1-15 (every integer)
  Medium: 17, 19, 21, 23, 25, 27, 29, 31, 35, 39, 43, 47
  Large: 51, 59, 67, 75, 83, 99, 115, 131, 147, 163, 179, 195

KEY EVALUATION METRICS:
----------------------
For each (metric, k) combination:
  - VR Accuracy: Attack success on victim records (should be high)
  - BR Accuracy: Baseline on test records (should be lower)
  - Delta = |VR Accuracy - BR Accuracy|: Information leakage indicator

SATURATION ANALYSIS:
-------------------
For each distance metric, we identify the SATURATION POINT:
  - The smallest k where adding more neighbors doesn't improve accuracy
  - Defined as: improvement < 0.005 over 3 consecutive k values
  - Helps select optimal k without overfitting

OUTPUTS:
--------
1. Complete results table (CSV):
   - All metrics × all k values
   - VR accuracy, BR accuracy, delta

2. Saturation analysis (CSV):
   - Saturation k for each metric
   - Maximum accuracy achieved
   - Minimum delta (privacy leakage measure)

3. Visualization Plots:
   a) VR Accuracy vs k (top 10 metrics)
   b) BR Accuracy vs k (top 10 metrics)
   c) Delta vs k with green (5%) and yellow (10%) thresholds
   d) Saturation analysis bar chart
   e) Heatmap of accuracies across metrics × k

INTERPRETATION GUIDELINES:
-------------------------
SUCCESSFUL ATTACK:
  - VR accuracy significantly > BR accuracy
  - Delta > 0.05 (green threshold)
  - Top metrics achieve VR accuracy > 0.40-0.50

MODERATE LEAKAGE:
  - Delta between 0.03 and 0.05
  - Attack works but limited effectiveness

GOOD PRIVACY:
  - Delta < 0.03
  - VR accuracy close to BR accuracy
  - Both near random baseline (0.25)

DATA PREPROCESSING:
------------------
This pipeline applies:
1. City name → Geographic distance (km from Puglia centroid)
2. One-hot encoding for categorical variables
3. Label encoding for non-Gower metrics
4. Special handling for binary metrics (boolean conversion)

COMPARISON WITH ATTACK MODEL 2:
-------------------------------
Attack Model 2 (XGBoost + meta-features): More complex, potentially higher accuracy
Attack Model 3 (k-NN + distance metrics): Simpler, interpretable, baseline comparison

================================================================================

In [ ]:
# ==========================================================
# CONFIGURATION
# ==========================================================
class Config:
    # Randomization settings
    R = 'Y'              # 'Y' for randomizzazione (RY), 'N' for no randomizzazione (RN)
    PR = 40              # Randomization percentage: 0, 10, 20, 30, 40, 100
    SYNTHESIZER_TYPE = 'CTGAN'  # CTGAN, TVAE, XGBoost_O0
    USERNAME = 'donatella.papa'
    
    # Derived config name
    RANDOMIZATION_LABEL = f'RY_PR{PR}' if R == 'Y' else 'RN_PR0'
    
    # Base paths
    BASE_PATH = rf'C:\Users\{USERNAME}\OneDrive - ISTAT\WP_13_Istat_Eurostat'
    
    # Input paths
    INPUT_REAL = f'{BASE_PATH}\\Step1\\Output'
    INPUT_SYNTH = f'{BASE_PATH}\\Step2\\Output'
    INPUT_COMUNI = f'{BASE_PATH}\\Step4\\Input\\gi_comuni.csv'
    
    # Output paths
    OUTPUT_REPORT = f'{BASE_PATH}\\Step4\\Output\\Report'
    OUTPUT_PLOT = f'{BASE_PATH}\\Step4\\Output\\Plot'
    
    # Extended k values up to ~200
    K_VALUES = [
        1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15,
        17, 19, 21, 23, 25, 27, 29, 31, 35, 39, 43, 47,
        51, 59, 67, 75, 83, 99, 115, 131, 147, 163, 179, 195
    ]
    
    # All pairwise metrics
    METRICS = [
        'l2', 'sqeuclidean', 'manhattan', 'l1', 'cityblock',
        'cosine', 'braycurtis', 'nan_euclidean', 'correlation', 
        'hamming', 'euclidean', 'canberra'
    ]
    
    BINARY_METRICS = ['sokalmichener', 'sokalsneath', 'yule', 'russellrao', 'dice', 'rogerstanimoto', 'jaccard']

# ==========================================================
# DATA LOADING
# ==========================================================
def load_data(cfg):
    # Build file paths
    real_path = rf'{cfg.INPUT_REAL}\real_data_datasetM10_TH20_R{cfg.R}_PR{cfg.PR}_4CAT_MISS_X2.csv'
    unseen_path = rf'{cfg.INPUT_REAL}\real_data_datasetM2_TH20_R{cfg.R}_PR{cfg.PR}_4CAT_MISS_X2.csv'
    synth_path = rf'{cfg.INPUT_SYNTH}\synthetic_data_datasetM10_TH20_R{cfg.R}_PR{cfg.PR}_4CAT_MISS_X2_{cfg.SYNTHESIZER_TYPE}.csv'
    
    print(f"Loading real data from: {real_path}")
    print(f"Loading unseen data from: {unseen_path}")
    print(f"Loading synth data from: {synth_path}")
    
    real = pd.read_csv(real_path)
    unseen = pd.read_csv(unseen_path)
    synth = pd.read_csv(synth_path)
    
    # List of columns to drop (only those that exist in the dataframe)
    cols_to_drop = ['id', 'score1', 'score2', 'bin1', 'bin2', 'raw_class', 
                    'age_norm', 'activity_norm', 'predisposition_norm', 
                    'birth_year', 'birth_month', 'birth_day', 
                    'municipality_birth', 'birth_dayofyear']
    
    # Drop only columns that exist
    real_cols_to_drop = [col for col in cols_to_drop if col in real.columns]
    unseen_cols_to_drop = [col for col in cols_to_drop if col in unseen.columns]
    synth_cols_to_drop = [col for col in cols_to_drop if col in synth.columns]
    
    if real_cols_to_drop:
        real = real.drop(columns=real_cols_to_drop)
    if unseen_cols_to_drop:
        unseen = unseen.drop(columns=unseen_cols_to_drop)
    if synth_cols_to_drop:
        synth = synth.drop(columns=synth_cols_to_drop)
    
    return real, unseen, synth

# ==========================================================
# CITY DISTANCE TRANSFORMATION
# ==========================================================
def load_city_data(cfg):
    import unicodedata
    def normalize(text):
        if pd.isna(text):
            return text
        text = str(text).strip().upper()
        return unicodedata.normalize('NFKD', text).encode('ascii', errors='ignore').decode('utf-8')
    
    cities = pd.read_csv(cfg.INPUT_COMUNI, sep=";")
    cities['den_norm'] = cities['denominazione_ita'].apply(normalize)
    cities = cities[~((cities['den_norm'] == "CASTRO") & (cities['sigla_provincia'] != "LE"))]
    return cities.drop_duplicates(subset='den_norm')

def distance_from_puglia(city_name, cities):
    import unicodedata
    def normalize(text):
        if pd.isna(text):
            return text
        text = str(text).strip().upper()
        return unicodedata.normalize('NFKD', text).encode('ascii', errors='ignore').decode('utf-8')
    
    city_norm = normalize(city_name)
    row = cities[cities['den_norm'] == city_norm]
    if row.empty:
        return np.nan
    
    lat = float(str(row.iloc[0]['lat']).replace(',', '.'))
    lon = float(str(row.iloc[0]['lon']).replace(',', '.'))
    lat_ctr, lon_ctr = 41.25, 16.25
    
    R = 6371
    lat_rad, lon_rad = np.radians(lat), np.radians(lon)
    lat_ctr_rad, lon_ctr_rad = np.radians(lat_ctr), np.radians(lon_ctr)
    
    a = np.sin((lat_rad - lat_ctr_rad)/2)**2 + np.cos(lat_ctr_rad) * np.cos(lat_rad) * np.sin((lon_rad - lon_ctr_rad)/2)**2
    return int(round(R * 2 * np.arcsin(np.sqrt(a))))

def transform_cities(df, cities):
    df = df.copy()
    df['municipality_residence'] = df['municipality_residence'].apply(lambda x: distance_from_puglia(x, cities))
    for col in ['physical_activity', 'genetic_predisposition']:
        df[col] = pd.to_numeric(df[col].replace('Missing', 0), errors='coerce')
    return df

# ==========================================================
# EVALUATION FUNCTIONS
# ==========================================================
def prepare_for_gower(df):
    df = df.copy()
    df.replace('Missing', np.nan, inplace=True)
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].astype(float)
        else:
            df[col] = df[col].astype(object)
    return df

def majority_vote_weighted(y_neighbors, distances, epsilon=1e-5):
    weights = 1 / (distances + epsilon)
    classes = np.unique(y_neighbors)
    class_weights = np.array([weights[y_neighbors == c].sum() for c in classes])
    
    if len(class_weights) == 0 or np.all(np.isnan(class_weights)):
        return np.random.choice(y_neighbors), False
    
    max_weight = np.nanmax(class_weights)
    candidates = classes[class_weights == max_weight]
    return np.random.choice(candidates), len(candidates) > 1

def evaluate_knn(X_train, y_train, X_test, y_test, k_values, metric='gower', is_binary=False):
    results = []
    
    if metric == 'gower':
        X_train_prep = prepare_for_gower(X_train)
        X_test_prep = prepare_for_gower(X_test)
        dist_matrix = gower.gower_matrix(X_test_prep, X_train_prep)
    else:
        if is_binary:
            X_test_input = X_test.astype(bool)
            X_train_input = X_train.astype(bool)
        else:
            X_test_input = X_test.values
            X_train_input = X_train.values
        dist_matrix = pairwise_distances(X_test_input, X_train_input, metric=metric)
    
    for k in k_values:
        if k > len(X_train):
            continue
        indices = np.argsort(dist_matrix, axis=1)[:, :k]
        predictions = []
        
        for i in range(len(X_test)):
            neighbor_vals = y_train.iloc[indices[i]].values
            neighbor_dists = dist_matrix[i, indices[i]]
            pred, _ = majority_vote_weighted(neighbor_vals, neighbor_dists)
            predictions.append(pred)
        
        results.append({'k': k, 'accuracy': accuracy_score(y_test, predictions)})
    
    return results

# ==========================================================
# SATURATION ANALYSIS
# ==========================================================
def find_saturation_k(results_df, threshold=0.005):
    df_sorted = results_df.sort_values('k')
    improvements = []
    
    for i in range(1, len(df_sorted)):
        prev_acc = df_sorted['accuracy'].iloc[i-1]
        curr_acc = df_sorted['accuracy'].iloc[i]
        improvements.append(curr_acc - prev_acc)
    
    window = 3
    for i in range(len(improvements) - window + 1):
        avg_improve = np.mean(improvements[i:i+window])
        if avg_improve < threshold:
            return df_sorted['k'].iloc[i + window - 1]
    
    return df_sorted['k'].iloc[-1]

# ==========================================================
# PLOTTING FUNCTIONS
# ==========================================================
def setup_plot_style():
    plt.style.use('seaborn-v0_8-darkgrid')
    sns.set_palette("husl")
    plt.rcParams['font.size'] = 12
    plt.rcParams['axes.labelsize'] = 14
    plt.rcParams['axes.titlesize'] = 16
    plt.rcParams['xtick.labelsize'] = 12
    plt.rcParams['ytick.labelsize'] = 12
    plt.rcParams['legend.fontsize'] = 11
    plt.rcParams['figure.titlesize'] = 18

def plot_top_models_accuracy(results_df, saturation_df, cfg, plot_path):
    """Plot accuracy vs k for top models"""
    setup_plot_style()
    
    top_models = saturation_df.nlargest(10, 'max_acc_real')['model'].tolist()
    model_labels = [m.replace('MM_', '').replace('GOWER', 'Gower') for m in top_models]
    # === GRAYSCALE SETTINGS ===
    grays = np.linspace(0.1, 0.8, len(top_models))
    colors = [str(g) for g in grays]

    markers = ['o', 's', '^', 'D', 'v', 'P', 'X', '*', '<', '>']
    linestyles = ['-', '--', '-.', ':']
    
    # Real data plot
    fig, ax = plt.subplots(figsize=(14, 8))
    
    for idx, model in enumerate(top_models):
        df_model = results_df[results_df['model'] == model].sort_values('k')
        ax.plot(df_model['k'],
                df_model['acc_real'],
                marker=markers[idx % len(markers)],
                linestyle=linestyles[idx % len(linestyles)],
                color=colors[idx],
                label=model_labels[idx],
                linewidth=2,markersize=5, markeredgecolor='black',markeredgewidth=0.8)
    
    ax.set_xlabel('k', fontsize=14, fontweight='bold')
    ax.set_ylabel('VR Accuracy', fontsize=14, fontweight='bold')
    #ax.set_title(f'Accuracy vs k - VR\n{cfg.SYNTHESIZER_TYPE} - {cfg.RANDOMIZATION_LABEL}', 
    #             fontsize=16, fontweight='bold')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0.25)
    
    plt.tight_layout()
    plt.subplots_adjust(right=0.75)
    plt.savefig(plot_path / f'accuracy_vs_k_VR_{cfg.SYNTHESIZER_TYPE}_{cfg.RANDOMIZATION_LABEL}.jpeg', 
                dpi=300, bbox_inches='tight')
    plt.close()
    
    # Unseen data plot
    fig, ax = plt.subplots(figsize=(14, 8))
    
    for idx, model in enumerate(top_models):
        df_model = results_df[results_df['model'] == model].sort_values('k')
        ax.plot(
    df_model['k'],
    df_model['acc_unseen'],
    marker=markers[idx % len(markers)],
    linestyle=linestyles[idx % len(linestyles)],
    color=colors[idx],
    label=model_labels[idx],
    linewidth=2,
    markersize=5,
    markeredgecolor='black',
    markeredgewidth=0.8
)
    
    ax.set_xlabel('k', fontsize=14, fontweight='bold')
    ax.set_ylabel('BR Accuracy', fontsize=14, fontweight='bold')
    #ax.set_title(f'Accuracy vs k - BR\n{cfg.SYNTHESIZER_TYPE} - {cfg.RANDOMIZATION_LABEL}', 
    #             fontsize=16, fontweight='bold')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0.25)
    
    plt.tight_layout()
    plt.subplots_adjust(right=0.75)
    plt.savefig(plot_path / f'accuracy_vs_k_BR_{cfg.SYNTHESIZER_TYPE}_{cfg.RANDOMIZATION_LABEL}.jpeg', 
                dpi=300, bbox_inches='tight')
    plt.close()

def plot_delta_analysis(results_df, saturation_df, cfg, plot_path):
    """Plot delta (difference between VR and BR)"""
    setup_plot_style()
    
    top_models = saturation_df.nlargest(8, 'max_acc_real')['model'].tolist()
    model_labels = [m.replace('MM_', '').replace('GOWER', 'Gower') for m in top_models]
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(top_models)))
    
    for idx, model in enumerate(top_models):
        df_model = results_df[results_df['model'] == model].sort_values('k')
        ax.plot(df_model['k'], df_model['delta'], 'o-', 
                color=colors[idx], label=model_labels[idx], 
                linewidth=2, markersize=5)
    
    ax.axhline(y=0.05, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Green threshold (0.05)')
    ax.axhline(y=0.10, color='orange', linestyle='--', linewidth=2, alpha=0.7, label='Yellow threshold (0.10)')
    
    ax.set_xlabel('k', fontsize=14, fontweight='bold')
    ax.set_ylabel('Delta |VR - BR|', fontsize=14, fontweight='bold')
    ax.set_title(f'Delta vs k - VR vs BR Difference\n{cfg.SYNTHESIZER_TYPE} - {cfg.RANDOMIZATION_LABEL}', 
                 fontsize=16, fontweight='bold')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0, top=0.2)
    
    plt.tight_layout()
    plt.subplots_adjust(right=0.75)
    plt.savefig(plot_path / f'delta_vs_k_{cfg.SYNTHESIZER_TYPE}_{cfg.RANDOMIZATION_LABEL}.jpeg', 
                dpi=300, bbox_inches='tight')
    plt.close()

def plot_saturation_summary(saturation_df, cfg, plot_path):
    """Plot saturation k for top models"""
    setup_plot_style()
    
    top_models = saturation_df.nlargest(15, 'max_acc_real')
    model_labels = [m.replace('MM_', '').replace('GOWER', 'Gower')[:20] for m in top_models['model']]
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    x = np.arange(len(top_models))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, top_models['sat_k'], width, 
                   label='Saturation k', color='steelblue', alpha=0.8)
    bars2 = ax.bar(x + width/2, top_models['k_max_real'], width, 
                   label='k at Max Accuracy', color='coral', alpha=0.8)
    
    ax.set_xlabel('Model', fontsize=14, fontweight='bold')
    ax.set_ylabel('k', fontsize=14, fontweight='bold')
    ax.set_title(f'Saturation vs Max Accuracy k - Top 15 Models\n{cfg.SYNTHESIZER_TYPE} - {cfg.RANDOMIZATION_LABEL}', 
                 fontsize=16, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(model_labels, rotation=45, ha='right', fontsize=10)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(plot_path / f'saturation_analysis_{cfg.SYNTHESIZER_TYPE}_{cfg.RANDOMIZATION_LABEL}.jpeg', 
                dpi=300, bbox_inches='tight')
    plt.close()

def plot_comparative_heatmap(results_df, cfg, plot_path):
    """Create heatmap of accuracy across models and k values"""
    setup_plot_style()
    
    # Pivot table for heatmap (select subset of models and k)
    top_models = results_df.groupby('model')['acc_real'].max().nlargest(12).index.tolist()
    selected_k = [3, 5, 7, 9, 11, 13, 19, 27, 39, 51, 75, 99, 131, 163, 195]
    
    df_filtered = results_df[results_df['model'].isin(top_models) & results_df['k'].isin(selected_k)]
    pivot_real = df_filtered.pivot(index='model', columns='k', values='acc_real')
    
    # Clean labels
    pivot_real.index = [idx.replace('MM_', '').replace('GOWER', 'Gower') for idx in pivot_real.index]
    
    fig, ax = plt.subplots(figsize=(16, 10))
    
    sns.heatmap(pivot_real, annot=True, fmt='.3f', cmap='YlOrRd', 
                linewidths=0.5, ax=ax, cbar_kws={'label': 'VR Accuracy'})
    
    ax.set_title(f'VR Accuracy Heatmap - Models vs k\n{cfg.SYNTHESIZER_TYPE} - {cfg.RANDOMIZATION_LABEL}', 
                 fontsize=16, fontweight='bold')
    ax.set_xlabel('k', fontsize=14, fontweight='bold')
    ax.set_ylabel('Model', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(plot_path / f'accuracy_heatmap_{cfg.SYNTHESIZER_TYPE}_{cfg.RANDOMIZATION_LABEL}.jpeg', 
                dpi=300, bbox_inches='tight')
    plt.close()

# ==========================================================
# MAIN PIPELINE
# ==========================================================
def run_pipeline():
    cfg = Config()
    
    # Create output directories
    report_path = Path(cfg.OUTPUT_REPORT)
    plot_path = Path(cfg.OUTPUT_PLOT)
    report_path.mkdir(parents=True, exist_ok=True)
    plot_path.mkdir(parents=True, exist_ok=True)
    
    print(f"Pipeline started")
    print(f"  Synthesizer: {cfg.SYNTHESIZER_TYPE}")
    print(f"  Randomization: {cfg.RANDOMIZATION_LABEL}")
    print(f"  K values: {len(cfg.K_VALUES)} (max {max(cfg.K_VALUES)})")
    print(f"  Metrics: {len(cfg.METRICS)} + GOWER")
    print(f"  Report output: {report_path}")
    print(f"  Plot output: {plot_path}")
    print("-" * 60)
    
    # Load and prepare data
    df_real, df_unseen, df_synth = load_data(cfg)
    cities = load_city_data(cfg)
    
    df_real = transform_cities(df_real, cities)
    df_unseen = transform_cities(df_unseen, cities)
    df_synth = transform_cities(df_synth, cities)
    
    X_synth, y_synth = df_synth.drop(columns=['diagnosis']), df_synth['diagnosis']
    X_real, y_real = df_real.drop(columns=['diagnosis']), df_real['diagnosis']
    X_unseen, y_unseen = df_unseen.drop(columns=['diagnosis']), df_unseen['diagnosis']
    
    # Label encoding for non-Gower metrics
    encoders = {}
    for col in X_synth.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        X_synth[col] = le.fit_transform(X_synth[col].astype(str))
        encoders[col] = le
    
    for df in [X_real, X_unseen]:
        for col in encoders:
            df[col] = df[col].map(lambda x: encoders[col].transform([str(x)])[0] if str(x) in encoders[col].classes_ else -1)
    
    # Storage for all results
    all_results = []
    
    # GOWER evaluation
    print("Evaluating GOWER...")
    gower_real = evaluate_knn(X_synth, y_synth, X_real, y_real, cfg.K_VALUES, metric='gower')
    gower_unseen = evaluate_knn(X_synth, y_synth, X_unseen, y_unseen, cfg.K_VALUES, metric='gower')
    
    gower_dict_real = {r['k']: r['accuracy'] for r in gower_real}
    gower_dict_unseen = {r['k']: r['accuracy'] for r in gower_unseen}
    
    for k in cfg.K_VALUES:
        if k in gower_dict_real and k in gower_dict_unseen:
            acc_real = gower_dict_real[k]
            acc_unseen = gower_dict_unseen[k]
            delta = abs(acc_real - acc_unseen)
            all_results.append({
                'model': 'GOWER',
                'k': k,
                'acc_real': acc_real,
                'acc_unseen': acc_unseen,
                'delta': delta
            })
    
    # Multi-metric evaluation
    for metric in cfg.METRICS:
        print(f"Evaluating {metric}...")
        is_binary = metric in cfg.BINARY_METRICS
        
        real_res = evaluate_knn(X_synth, y_synth, X_real, y_real, cfg.K_VALUES, metric=metric, is_binary=is_binary)
        unseen_res = evaluate_knn(X_synth, y_synth, X_unseen, y_unseen, cfg.K_VALUES, metric=metric, is_binary=is_binary)
        
        real_dict = {r['k']: r['accuracy'] for r in real_res}
        unseen_dict = {r['k']: r['accuracy'] for r in unseen_res}
        
        for k in cfg.K_VALUES:
            if k in real_dict and k in unseen_dict:
                acc_real = real_dict[k]
                acc_unseen = unseen_dict[k]
                delta = abs(acc_real - acc_unseen)
                all_results.append({
                    'model': f'MM_{metric}',
                    'k': k,
                    'acc_real': acc_real,
                    'acc_unseen': acc_unseen,
                    'delta': delta
                })
    
    # Create final dataframe
    results_df = pd.DataFrame(all_results)
    
    # Calculate saturation for each model
    saturation_results = []
    for model in results_df['model'].unique():
        df_model = results_df[results_df['model'] == model].copy()
        if len(df_model) > 5:
            sat_k = find_saturation_k(df_model[['k', 'acc_real']].rename(columns={'acc_real': 'accuracy'}))
            max_acc = df_model['acc_real'].max()
            max_unseen = df_model['acc_unseen'].max()
            max_k = df_model.loc[df_model['acc_real'].idxmax(), 'k']
            min_delta = df_model['delta'].min()
            
            saturation_results.append({
                'model': model,
                'sat_k': sat_k,
                'max_acc_real': max_acc,
                'max_acc_unseen': max_unseen,
                'k_max_real': max_k,
                'min_delta': min_delta
            })
    
    saturation_df = pd.DataFrame(saturation_results).sort_values('max_acc_real', ascending=False)
    
    # Save CSV results to Report folder
    results_df.to_csv(report_path / f'full_report_{cfg.SYNTHESIZER_TYPE}_{cfg.RANDOMIZATION_LABEL}.csv', index=False)
    saturation_df.to_csv(report_path / f'saturation_{cfg.SYNTHESIZER_TYPE}_{cfg.RANDOMIZATION_LABEL}.csv', index=False)
    
    # Generate plots to Plot folder
    print("-" * 60)
    print("Generating plots...")
    
    plot_top_models_accuracy(results_df, saturation_df, cfg, plot_path)
    plot_delta_analysis(results_df, saturation_df, cfg, plot_path)
    plot_saturation_summary(saturation_df, cfg, plot_path)
    plot_comparative_heatmap(results_df, cfg, plot_path)
    
    # Print summary
    print("-" * 60)
    print("Summary Results:")
    print(f"  Top model by VR accuracy: {saturation_df.iloc[0]['model']} ({saturation_df.iloc[0]['max_acc_real']:.4f})")
    print(f"  Top model by BR accuracy: {saturation_df.loc[saturation_df['max_acc_unseen'].idxmax(), 'model']} ({saturation_df['max_acc_unseen'].max():.4f})")
    print(f"  Minimum delta achieved: {saturation_df['min_delta'].min():.4f}")
    print(f"  Average saturation k: {saturation_df['sat_k'].mean():.1f}")
    print(f"\nReports saved to: {report_path}")
    print(f"Plots saved to: {plot_path}")
    
    return results_df, saturation_df

# ==========================================================
# EXECUTE
# ==========================================================
if __name__ == "__main__":
    start = time.time()
    results, saturation = run_pipeline()
    print(f"\nTotal execution time: {time.time() - start:.1f}s")

Pipeline started
  Synthesizer: CTGAN
  Randomization: RY_PR40
  K values: 39 (max 195)
  Metrics: 12 + GOWER
  Report output: C:\Users\donatella.papa\OneDrive - ISTAT\WP_13_Istat_Eurostat\Step4\Output\Report
  Plot output: C:\Users\donatella.papa\OneDrive - ISTAT\WP_13_Istat_Eurostat\Step4\Output\Plot
------------------------------------------------------------
Loading real data from: C:\Users\donatella.papa\OneDrive - ISTAT\WP_13_Istat_Eurostat\Step1\Output\real_data_datasetM10_TH20_RY_PR40_4CAT_MISS_X2.csv
Loading unseen data from: C:\Users\donatella.papa\OneDrive - ISTAT\WP_13_Istat_Eurostat\Step1\Output\real_data_datasetM2_TH20_RY_PR40_4CAT_MISS_X2.csv
Loading synth data from: C:\Users\donatella.papa\OneDrive - ISTAT\WP_13_Istat_Eurostat\Step2\Output\synthetic_data_datasetM10_TH20_RY_PR40_4CAT_MISS_X2_CTGAN.csv
Evaluating GOWER...
Evaluating l2...
Evaluating sqeuclidean...
Evaluating manhattan...
Evaluating l1...
Evaluating cityblock...
Evaluating cosine...
Evaluating braycurtis..

In [13]:
# ==========================================================
# ONLY PLOTS - NO CALCULATIONS (WITHOUT TITLES)
# ==========================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ==========================================================
# CONFIGURATION (must match original)
# ==========================================================
class Config:
    # Randomization settings
    R = 'Y'              # 'Y' for randomizzazione (RY), 'N' for no randomizzazione (RN)
    PR = 40              # Randomization percentage: 0, 10, 20, 30, 40, 100
    SYNTHESIZER_TYPE = 'XGBoost_O0'  # CTGAN, TVAE, XGBoost_O0
    USERNAME = 'donatella.papa'
    
    # Derived config name
    RANDOMIZATION_LABEL = f'RY_PR{PR}' if R == 'Y' else 'RN_PR0'
    
    # Base paths
    BASE_PATH = rf'C:\Users\{USERNAME}\OneDrive - ISTAT\WP_13_Istat_Eurostat'
    
    # Output paths
    OUTPUT_REPORT = f'{BASE_PATH}\\Step4\\Output\\Report\\attack1_used'
    OUTPUT_PLOT = f'{BASE_PATH}\\Step4\\Output\\Plot'

# ==========================================================
# CUSTOM COLOR PALETTES
# ==========================================================

# Option 1: Professional color palette (blue/orange/teal/purple/red)
COLOR_PALETTE_1 = [
    '#2E86AB',  # Blue
    '#A23B72',  # Purple
    '#F18F01',  # Orange
    '#C73E1D',  # Red
    '#6A994E',  # Green
    '#BC4A6C',  # Pink
    '#1C7C54',  # Dark Green
    '#D62828',  # Bright Red
    '#003D5B',  # Navy
    '#E09F3E'   # Gold
]

# Choose which palette to use
ACTIVE_PALETTE = COLOR_PALETTE_1

# ==========================================================
# PLOTTING FUNCTIONS (WITHOUT TITLES)
# ==========================================================

def setup_plot_style():
    plt.style.use('seaborn-v0_8-darkgrid')
    sns.set_palette("husl")
    plt.rcParams['font.size'] = 12
    plt.rcParams['axes.labelsize'] = 14
    plt.rcParams['axes.titlesize'] = 16
    plt.rcParams['xtick.labelsize'] = 12
    plt.rcParams['ytick.labelsize'] = 12
    plt.rcParams['legend.fontsize'] = 11
    plt.rcParams['figure.titlesize'] = 18

def plot_top_models_accuracy(results_df, saturation_df, cfg, plot_path):
    """Plot accuracy vs k for top models with colors - NO TITLES"""
    setup_plot_style()
    
    # Determine which column name to use for models
    model_col = 'modello' if 'modello' in saturation_df.columns else 'model'
    
    top_models = saturation_df.nlargest(10, 'max_acc_real')[model_col].tolist()
    model_labels = [m.replace('MM_', '').replace('GOWER', 'Gower') for m in top_models]
    
    # Use active color palette
    colors = ACTIVE_PALETTE[:len(top_models)]
    markers = ['o', 's', '^', 'D', 'v', 'P', 'X', '*', '<', '>']
    linestyles = ['-', '--', '-.', ':']
    
    # Determine column name for model in results_df
    results_model_col = 'modello' if 'modello' in results_df.columns else 'model'
    
    # Real data plot (VR) - NO TITLE
    fig, ax = plt.subplots(figsize=(14, 8))
    
    for idx, model in enumerate(top_models):
        df_model = results_df[results_df[results_model_col] == model].sort_values('k')
        ax.plot(df_model['k'],
                df_model['acc_real'],
                marker=markers[idx % len(markers)],
                linestyle=linestyles[idx % len(linestyles)],
                color=colors[idx],
                label=model_labels[idx],
                linewidth=2, markersize=6, markeredgecolor='black', markeredgewidth=0.8)
    
    ax.set_xlabel('k', fontsize=14, fontweight='bold')
    ax.set_ylabel('VR Accuracy', fontsize=14, fontweight='bold')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0.25)
    
    plt.tight_layout()
    plt.subplots_adjust(right=0.75)
    plt.savefig(plot_path / f'accuracy_vs_k_VR_{cfg.SYNTHESIZER_TYPE}_{cfg.RANDOMIZATION_LABEL}.jpeg', 
                dpi=300, bbox_inches='tight')
    plt.close()
    
    # Unseen data plot (BR) - NO TITLE
    fig, ax = plt.subplots(figsize=(14, 8))
    
    for idx, model in enumerate(top_models):
        df_model = results_df[results_df[results_model_col] == model].sort_values('k')
        ax.plot(df_model['k'],
                df_model['acc_unseen'],
                marker=markers[idx % len(markers)],
                linestyle=linestyles[idx % len(linestyles)],
                color=colors[idx],
                label=model_labels[idx],
                linewidth=2, markersize=6, markeredgecolor='black', markeredgewidth=0.8)
    
    ax.set_xlabel('k', fontsize=14, fontweight='bold')
    ax.set_ylabel('BR Accuracy', fontsize=14, fontweight='bold')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0.25)
    
    plt.tight_layout()
    plt.subplots_adjust(right=0.75)
    plt.savefig(plot_path / f'accuracy_vs_k_BR_{cfg.SYNTHESIZER_TYPE}_{cfg.RANDOMIZATION_LABEL}.jpeg', 
                dpi=300, bbox_inches='tight')
    plt.close()

def plot_delta_analysis(results_df, saturation_df, cfg, plot_path):
    """Plot delta (difference between VR and BR) - NO TITLE"""
    setup_plot_style()
    
    model_col = 'modello' if 'modello' in saturation_df.columns else 'model'
    results_model_col = 'modello' if 'modello' in results_df.columns else 'model'
    
    top_models = saturation_df.nlargest(8, 'max_acc_real')[model_col].tolist()
    model_labels = [m.replace('MM_', '').replace('GOWER', 'Gower') for m in top_models]
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Use plasma colormap for delta plot
    colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(top_models)))
    
    for idx, model in enumerate(top_models):
        df_model = results_df[results_df[results_model_col] == model].sort_values('k')
        ax.plot(df_model['k'], df_model['delta'], 'o-', 
                color=colors[idx], label=model_labels[idx], 
                linewidth=2, markersize=6)
    
    ax.axhline(y=0.05, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Green threshold (0.05)')
    ax.axhline(y=0.10, color='orange', linestyle='--', linewidth=2, alpha=0.7, label='Yellow threshold (0.10)')
    
    ax.set_xlabel('k', fontsize=14, fontweight='bold')
    ax.set_ylabel('Delta |VR - BR|', fontsize=14, fontweight='bold')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0, top=0.2)
    
    plt.tight_layout()
    plt.subplots_adjust(right=0.75)
    plt.savefig(plot_path / f'delta_vs_k_{cfg.SYNTHESIZER_TYPE}_{cfg.RANDOMIZATION_LABEL}.jpeg', 
                dpi=300, bbox_inches='tight')
    plt.close()

def plot_saturation_summary(saturation_df, cfg, plot_path):
    """Plot saturation k for top models - NO TITLE"""
    setup_plot_style()
    
    # Check what columns are available
    print("\nSaturation DataFrame columns:", list(saturation_df.columns))
    
    # Determine column names (adapt to what's available)
    model_col = 'modello' if 'modello' in saturation_df.columns else 'model'
    
    # Check for saturation column (might be 'sat_k' or 'saturation_k' or similar)
    sat_col = None
    for possible_name in ['sat_k', 'saturation_k', 'k_saturation', 'saturation']:
        if possible_name in saturation_df.columns:
            sat_col = possible_name
            break
    
    if sat_col is None:
        print(f"WARNING: No saturation column found. Available columns: {list(saturation_df.columns)}")
        print("Creating saturation column from data...")
        # If no saturation column, we can't plot this
        return
    
    # Check for k_max column
    kmax_col = None
    for possible_name in ['k_max_real', 'k_max', 'max_k', 'k_at_max_accuracy']:
        if possible_name in saturation_df.columns:
            kmax_col = possible_name
            break
    
    if kmax_col is None:
        print(f"WARNING: No k_max column found. Available columns: {list(saturation_df.columns)}")
        return
    
    top_models = saturation_df.nlargest(15, 'max_acc_real')
    model_labels = [m.replace('MM_', '').replace('GOWER', 'Gower')[:20] for m in top_models[model_col]]
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    x = np.arange(len(top_models))
    width = 0.35
    
    # Use active palette colors
    ax.bar(x - width/2, top_models[sat_col], width, 
           label='Saturation k', color=ACTIVE_PALETTE[0], alpha=0.8)
    ax.bar(x + width/2, top_models[kmax_col], width, 
           label='k at Max Accuracy', color=ACTIVE_PALETTE[1], alpha=0.8)
    
    ax.set_xlabel('Model', fontsize=14, fontweight='bold')
    ax.set_ylabel('k', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(model_labels, rotation=45, ha='right', fontsize=10)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(plot_path / f'saturation_analysis_{cfg.SYNTHESIZER_TYPE}_{cfg.RANDOMIZATION_LABEL}.jpeg', 
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saturation plot saved (using columns: {sat_col} and {kmax_col})")

def plot_comparative_heatmap(results_df, cfg, plot_path):
    """Create heatmap of accuracy across models and k values - NO TITLE"""
    setup_plot_style()
    
    # Determine column name for model
    model_col = 'modello' if 'modello' in results_df.columns else 'model'
    
    # Pivot table for heatmap (select subset of models and k)
    top_models = results_df.groupby(model_col)['acc_real'].max().nlargest(12).index.tolist()
    selected_k = [3, 5, 7, 9, 11, 13, 19, 27, 39, 51, 75, 99, 131, 163, 195]
    
    # Filter available k values
    available_k = [k for k in selected_k if k in results_df['k'].values]
    
    df_filtered = results_df[results_df[model_col].isin(top_models) & results_df['k'].isin(available_k)]
    pivot_real = df_filtered.pivot(index=model_col, columns='k', values='acc_real')
    
    # Clean labels
    pivot_real.index = [idx.replace('MM_', '').replace('GOWER', 'Gower') for idx in pivot_real.index]
    
    fig, ax = plt.subplots(figsize=(16, 10))
    
    # Custom colormap for heatmap
    sns.heatmap(pivot_real, annot=True, fmt='.3f', cmap='RdYlBu', 
                linewidths=0.5, ax=ax, cbar_kws={'label': 'VR Accuracy'},
                annot_kws={'size': 10})
    
    ax.set_xlabel('k', fontsize=14, fontweight='bold')
    ax.set_ylabel('Model', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(plot_path / f'accuracy_heatmap_{cfg.SYNTHESIZER_TYPE}_{cfg.RANDOMIZATION_LABEL}.jpeg', 
                dpi=300, bbox_inches='tight')
    plt.close()

# ==========================================================
# MAIN FUNCTION - ONLY LOADS DATA AND PLOTS
# ==========================================================

def generate_plots_only():
    """Load previously saved CSV results and generate plots only - WITHOUT TITLES"""
    
    cfg = Config()
    
    # Create output directories
    report_path = Path(cfg.OUTPUT_REPORT)
    plot_path = Path(cfg.OUTPUT_PLOT)
    plot_path.mkdir(parents=True, exist_ok=True)
    
    print("=" * 60)
    print("PLOT GENERATION ONLY - No calculations (Titles REMOVED)")
    print("=" * 60)
    print(f"Synthesizer: {cfg.SYNTHESIZER_TYPE}")
    print(f"Randomization: {cfg.RANDOMIZATION_LABEL}")
    print(f"Report path: {report_path}")
    print(f"Plot output: {plot_path}")
    print("-" * 60)
    
    # Load previously saved results with CORRECTED file names
    results_file = report_path / f'report_unificato_completo_{cfg.RANDOMIZATION_LABEL}_{cfg.SYNTHESIZER_TYPE}.csv'
    saturation_file = report_path / f'saturazione_modelli_{cfg.RANDOMIZATION_LABEL}_{cfg.SYNTHESIZER_TYPE}.csv'
    
    print(f"Looking for results file: {results_file.name}")
    print(f"Looking for saturation file: {saturation_file.name}")
    
    if not results_file.exists():
        print(f"\nERROR: Results file not found: {results_file}")
        print("\nAvailable files in directory:")
        if report_path.exists():
            csv_files = list(report_path.glob("*.csv"))
            if csv_files:
                for f in csv_files:
                    print(f"  - {f.name}")
            else:
                print("  No CSV files found!")
        else:
            print(f"  Directory does not exist: {report_path}")
        return None, None
    
    if not saturation_file.exists():
        print(f"\nERROR: Saturation file not found: {saturation_file}")
        return None, None
    
    print(f"\nLoading results from: {results_file.name}")
    results_df = pd.read_csv(results_file)
    
    print(f"Loading saturation from: {saturation_file.name}")
    saturation_df = pd.read_csv(saturation_file)
    
    # Display column names to verify
    print(f"\nResults DataFrame columns: {list(results_df.columns)}")
    print(f"Saturation DataFrame columns: {list(saturation_df.columns)}")
    
    print(f"\nLoaded {len(results_df)} result rows")
    print(f"Loaded {len(saturation_df)} model saturations")
    print("-" * 60)
    
    # Generate all plots
    print("Generating plots WITHOUT titles...")
    
    plot_top_models_accuracy(results_df, saturation_df, cfg, plot_path)
    print("  ✓ Accuracy plots (VR + BR) - no titles")
    
    plot_delta_analysis(results_df, saturation_df, cfg, plot_path)
    print("  ✓ Delta analysis plot - no title")
    
    plot_saturation_summary(saturation_df, cfg, plot_path)
    
    plot_comparative_heatmap(results_df, cfg, plot_path)
    print("  ✓ Comparative heatmap - no title")
    
    print("-" * 60)
    print(f"All plots saved to: {plot_path}")
    print("=" * 60)
    
    return results_df, saturation_df

# ==========================================================
# EXECUTE
# ==========================================================
if __name__ == "__main__":
    results, saturation = generate_plots_only()

PLOT GENERATION ONLY - No calculations (Titles REMOVED)
Synthesizer: XGBoost_O0
Randomization: RY_PR40
Report path: C:\Users\donatella.papa\OneDrive - ISTAT\WP_13_Istat_Eurostat\Step4\Output\Report\attack1_used
Plot output: C:\Users\donatella.papa\OneDrive - ISTAT\WP_13_Istat_Eurostat\Step4\Output\Plot
------------------------------------------------------------
Looking for results file: report_unificato_completo_RY_PR40_XGBoost_O0.csv
Looking for saturation file: saturazione_modelli_RY_PR40_XGBoost_O0.csv

Loading results from: report_unificato_completo_RY_PR40_XGBoost_O0.csv
Loading saturation from: saturazione_modelli_RY_PR40_XGBoost_O0.csv

Results DataFrame columns: ['modello', 'k', 'acc_real', 'acc_unseen', 'delta', 'fedelta', 'pct_tie_real', 'pct_tie_unseen']
Saturation DataFrame columns: ['modello', 'max_acc_real', 'k_max_real', 'max_acc_unseen', 'k_max_unseen', 'min_delta', 'sat_k_real', 'sat_k_unseen', 'tradeoff_k', 'tradeoff_acc', 'tradeoff_delta']

Loaded 507 result rows
Lo

In [23]:
# ==========================================================
# ONLY PLOTS - NO CALCULATIONS (WITHOUT TITLES) - EPS FORMAT
# ==========================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ==========================================================
# CONFIGURATION (must match original)
# ==========================================================
class Config:
    # Randomization settings
    R = 'Y'              # 'Y' for randomizzazione (RY), 'N' for no randomizzazione (RN)
    PR = 40              # Randomization percentage: 0, 10, 20, 30, 40, 100
    SYNTHESIZER_TYPE = 'CTGAN'  # CTGAN, TVAE, XGBoost_O0
    USERNAME = 'donatella.papa'
    
    # Derived config name
    RANDOMIZATION_LABEL = f'RY_PR{PR}' if R == 'Y' else 'RN_PR0'
    
    # Base paths
    BASE_PATH = rf'C:\Users\{USERNAME}\OneDrive - ISTAT\WP_13_Istat_Eurostat'
    
    # Output paths
    OUTPUT_REPORT = f'{BASE_PATH}\\Step4\\Output\\Report\\attack1_used'
    OUTPUT_PLOT = f'{BASE_PATH}\\Step4\\Output\\Plot'

# ==========================================================
# CUSTOM COLOR PALETTES
# ==========================================================

# Option 1: Professional color palette (blue/orange/teal/purple/red)
COLOR_PALETTE_1 = [
    '#2E86AB',  # Blue
    '#A23B72',  # Purple
    '#F18F01',  # Orange
    '#C73E1D',  # Red
    '#6A994E',  # Green
    '#BC4A6C',  # Pink
    '#1C7C54',  # Dark Green
    '#D62828',  # Bright Red
    '#003D5B',  # Navy
    '#E09F3E'   # Gold
]

# Choose which palette to use
ACTIVE_PALETTE = COLOR_PALETTE_1

# ==========================================================
# PLOTTING FUNCTIONS (WITHOUT TITLES) - EPS FORMAT
# ==========================================================

def setup_plot_style():
    """Setup plot style with white background"""
    # White background settings
    plt.rcParams['figure.facecolor'] = 'white'
    plt.rcParams['axes.facecolor'] = 'white'
    plt.rcParams['savefig.facecolor'] = 'white'
    
    # Font settings
    plt.rcParams['font.size'] = 12
    plt.rcParams['axes.labelsize'] = 14
    plt.rcParams['axes.titlesize'] = 16
    plt.rcParams['xtick.labelsize'] = 12
    plt.rcParams['ytick.labelsize'] = 12
    plt.rcParams['legend.fontsize'] = 11
    plt.rcParams['figure.titlesize'] = 18
    
    # Grid settings (like the other graph)
    plt.rcParams['axes.grid'] = True
    plt.rcParams['grid.alpha'] = 0.3
    plt.rcParams['grid.linestyle'] = '--'
    
    # For EPS export
    plt.rcParams['ps.useafm'] = True
    plt.rcParams['pdf.use14corefonts'] = True
    plt.rcParams['text.usetex'] = False

def plot_top_models_accuracy(results_df, saturation_df, cfg, plot_path):
    """Plot accuracy vs k for top models with colors - NO TITLES - EPS format"""
    setup_plot_style()
    
    # Determine which column name to use for models
    model_col = 'modello' if 'modello' in saturation_df.columns else 'model'
    
    top_models = saturation_df.nlargest(10, 'max_acc_real')[model_col].tolist()
    model_labels = [m.replace('MM_', '').replace('GOWER', 'Gower') for m in top_models]
    
    # Use active color palette
    colors = ACTIVE_PALETTE[:len(top_models)]
    markers = ['o', 's', '^', 'D', 'v', 'P', 'X', '*', '<', '>']
    linestyles = ['-', '--', '-.', ':']
    
    # Determine column name for model in results_df
    results_model_col = 'modello' if 'modello' in results_df.columns else 'model'
    
    # Real data plot (VR) - NO TITLE - EPS
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.set_facecolor('white')
    
    for idx, model in enumerate(top_models):
        df_model = results_df[results_df[results_model_col] == model].sort_values('k')
        ax.plot(df_model['k'],
                df_model['acc_real'],
                marker=markers[idx % len(markers)],
                linestyle=linestyles[idx % len(linestyles)],
                color=colors[idx],
                label=model_labels[idx],
                linewidth=2, markersize=6, markeredgecolor='black', markeredgewidth=0.8)
    
    ax.set_xlabel('k', fontsize=14, fontweight='bold')
    ax.set_ylabel('VR Accuracy', fontsize=14, fontweight='bold')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0.25)
    
    plt.tight_layout()
    plt.subplots_adjust(right=0.75)
    plt.savefig(plot_path / f'accuracy_vs_k_VR_{cfg.SYNTHESIZER_TYPE}_{cfg.RANDOMIZATION_LABEL}.eps', 
                format='eps', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    
    # Unseen data plot (BR) - NO TITLE - EPS
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.set_facecolor('white')
    
    for idx, model in enumerate(top_models):
        df_model = results_df[results_df[results_model_col] == model].sort_values('k')
        ax.plot(df_model['k'],
                df_model['acc_unseen'],
                marker=markers[idx % len(markers)],
                linestyle=linestyles[idx % len(linestyles)],
                color=colors[idx],
                label=model_labels[idx],
                linewidth=2, markersize=6, markeredgecolor='black', markeredgewidth=0.8)
    
    ax.set_xlabel('k', fontsize=14, fontweight='bold')
    ax.set_ylabel('BR Accuracy', fontsize=14, fontweight='bold')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0.25)
    
    plt.tight_layout()
    plt.subplots_adjust(right=0.75)
    plt.savefig(plot_path / f'accuracy_vs_k_BR_{cfg.SYNTHESIZER_TYPE}_{cfg.RANDOMIZATION_LABEL}.eps', 
                format='eps', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

def plot_delta_analysis(results_df, saturation_df, cfg, plot_path):
    """Plot delta (difference between VR and BR) - NO TITLE - EPS format"""
    setup_plot_style()
    
    model_col = 'modello' if 'modello' in saturation_df.columns else 'model'
    results_model_col = 'modello' if 'modello' in results_df.columns else 'model'
    
    top_models = saturation_df.nlargest(8, 'max_acc_real')[model_col].tolist()
    model_labels = [m.replace('MM_', '').replace('GOWER', 'Gower') for m in top_models]
    
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.set_facecolor('white')
    
    # Use plasma colormap for delta plot
    colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(top_models)))
    
    for idx, model in enumerate(top_models):
        df_model = results_df[results_df[results_model_col] == model].sort_values('k')
        ax.plot(df_model['k'], df_model['delta'], 'o-', 
                color=colors[idx], label=model_labels[idx], 
                linewidth=2, markersize=6)
    
    ax.axhline(y=0.05, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Green threshold (0.05)')
    ax.axhline(y=0.10, color='orange', linestyle='--', linewidth=2, alpha=0.7, label='Yellow threshold (0.10)')
    
    ax.set_xlabel('k', fontsize=14, fontweight='bold')
    ax.set_ylabel('Delta |VR - BR|', fontsize=14, fontweight='bold')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0, top=0.2)
    
    plt.tight_layout()
    plt.subplots_adjust(right=0.75)
    plt.savefig(plot_path / f'delta_vs_k_{cfg.SYNTHESIZER_TYPE}_{cfg.RANDOMIZATION_LABEL}.eps', 
                format='eps', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

def plot_saturation_summary(saturation_df, cfg, plot_path):
    """Plot saturation k for top models - NO TITLE - EPS format"""
    setup_plot_style()
    
    # Check what columns are available
    print("\nSaturation DataFrame columns:", list(saturation_df.columns))
    
    # Determine column names (adapt to what's available)
    model_col = 'modello' if 'modello' in saturation_df.columns else 'model'
    
    # Check for saturation column (might be 'sat_k' or 'saturation_k' or similar)
    sat_col = None
    for possible_name in ['sat_k', 'saturation_k', 'k_saturation', 'saturation']:
        if possible_name in saturation_df.columns:
            sat_col = possible_name
            break
    
    if sat_col is None:
        print(f"WARNING: No saturation column found. Available columns: {list(saturation_df.columns)}")
        print("Creating saturation column from data...")
        # If no saturation column, we can't plot this
        return
    
    # Check for k_max column
    kmax_col = None
    for possible_name in ['k_max_real', 'k_max', 'max_k', 'k_at_max_accuracy']:
        if possible_name in saturation_df.columns:
            kmax_col = possible_name
            break
    
    if kmax_col is None:
        print(f"WARNING: No k_max column found. Available columns: {list(saturation_df.columns)}")
        return
    
    top_models = saturation_df.nlargest(15, 'max_acc_real')
    model_labels = [m.replace('MM_', '').replace('GOWER', 'Gower')[:20] for m in top_models[model_col]]
    
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.set_facecolor('white')
    
    x = np.arange(len(top_models))
    width = 0.35
    
    # Use active palette colors
    ax.bar(x - width/2, top_models[sat_col], width, 
           label='Saturation k', color=ACTIVE_PALETTE[0], alpha=0.8)
    ax.bar(x + width/2, top_models[kmax_col], width, 
           label='k at Max Accuracy', color=ACTIVE_PALETTE[1], alpha=0.8)
    
    ax.set_xlabel('Model', fontsize=14, fontweight='bold')
    ax.set_ylabel('k', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(model_labels, rotation=45, ha='right', fontsize=10)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(plot_path / f'saturation_analysis_{cfg.SYNTHESIZER_TYPE}_{cfg.RANDOMIZATION_LABEL}.eps', 
                format='eps', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"  ✓ Saturation plot saved (using columns: {sat_col} and {kmax_col})")

def plot_comparative_heatmap(results_df, cfg, plot_path):
    """Create heatmap of accuracy across models and k values - NO TITLE - EPS format"""
    setup_plot_style()
    
    # Determine column name for model
    model_col = 'modello' if 'modello' in results_df.columns else 'model'
    
    # Pivot table for heatmap (select subset of models and k)
    top_models = results_df.groupby(model_col)['acc_real'].max().nlargest(12).index.tolist()
    selected_k = [3, 5, 7, 9, 11, 13, 19, 27, 39, 51, 75, 99, 131, 163, 195]
    
    # Filter available k values
    available_k = [k for k in selected_k if k in results_df['k'].values]
    
    df_filtered = results_df[results_df[model_col].isin(top_models) & results_df['k'].isin(available_k)]
    pivot_real = df_filtered.pivot(index=model_col, columns='k', values='acc_real')
    
    # Clean labels
    pivot_real.index = [idx.replace('MM_', '').replace('GOWER', 'Gower') for idx in pivot_real.index]
    
    fig, ax = plt.subplots(figsize=(16, 10))
    ax.set_facecolor('white')
    
    # Custom colormap for heatmap
    sns.heatmap(pivot_real, annot=True, fmt='.3f', cmap='RdYlBu', 
                linewidths=0.5, ax=ax, cbar_kws={'label': 'VR Accuracy'},
                annot_kws={'size': 10})
    
    ax.set_xlabel('k', fontsize=14, fontweight='bold')
    ax.set_ylabel('Model', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(plot_path / f'accuracy_heatmap_{cfg.SYNTHESIZER_TYPE}_{cfg.RANDOMIZATION_LABEL}.eps', 
                format='eps', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

# ==========================================================
# MAIN FUNCTION - ONLY LOADS DATA AND PLOTS
# ==========================================================

def generate_plots_only():
    """Load previously saved CSV results and generate plots only - WITHOUT TITLES - EPS format"""
    
    cfg = Config()
    
    # Create output directories
    report_path = Path(cfg.OUTPUT_REPORT)
    plot_path = Path(cfg.OUTPUT_PLOT)
    plot_path.mkdir(parents=True, exist_ok=True)
    
    print("=" * 60)
    print("PLOT GENERATION ONLY - No calculations (Titles REMOVED) - EPS FORMAT")
    print("=" * 60)
    print(f"Synthesizer: {cfg.SYNTHESIZER_TYPE}")
    print(f"Randomization: {cfg.RANDOMIZATION_LABEL}")
    print(f"Report path: {report_path}")
    print(f"Plot output: {plot_path}")
    print("-" * 60)
    
    # Load previously saved results with CORRECTED file names
    results_file = report_path / f'report_unificato_completo_{cfg.RANDOMIZATION_LABEL}_{cfg.SYNTHESIZER_TYPE}.csv'
    saturation_file = report_path / f'saturazione_modelli_{cfg.RANDOMIZATION_LABEL}_{cfg.SYNTHESIZER_TYPE}.csv'
    
    print(f"Looking for results file: {results_file.name}")
    print(f"Looking for saturation file: {saturation_file.name}")
    
    if not results_file.exists():
        print(f"\nERROR: Results file not found: {results_file}")
        print("\nAvailable files in directory:")
        if report_path.exists():
            csv_files = list(report_path.glob("*.csv"))
            if csv_files:
                for f in csv_files:
                    print(f"  - {f.name}")
            else:
                print("  No CSV files found!")
        else:
            print(f"  Directory does not exist: {report_path}")
        return None, None
    
    if not saturation_file.exists():
        print(f"\nERROR: Saturation file not found: {saturation_file}")
        return None, None
    
    print(f"\nLoading results from: {results_file.name}")
    results_df = pd.read_csv(results_file)
    
    print(f"Loading saturation from: {saturation_file.name}")
    saturation_df = pd.read_csv(saturation_file)
    
    # Display column names to verify
    print(f"\nResults DataFrame columns: {list(results_df.columns)}")
    print(f"Saturation DataFrame columns: {list(saturation_df.columns)}")
    
    print(f"\nLoaded {len(results_df)} result rows")
    print(f"Loaded {len(saturation_df)} model saturations")
    print("-" * 60)
    
    # Generate all plots
    print("Generating plots WITHOUT titles in EPS format...")
    
    plot_top_models_accuracy(results_df, saturation_df, cfg, plot_path)
    print("  ✓ Accuracy plots (VR + BR) - no titles - EPS")
    
    plot_delta_analysis(results_df, saturation_df, cfg, plot_path)
    print("  ✓ Delta analysis plot - no title - EPS")
    
    plot_saturation_summary(saturation_df, cfg, plot_path)
    
    plot_comparative_heatmap(results_df, cfg, plot_path)
    print("  ✓ Comparative heatmap - no title - EPS")
    
    print("-" * 60)
    print(f"All EPS plots saved to: {plot_path}")
    print("=" * 60)
    
    return results_df, saturation_df

# ==========================================================
# EXECUTE
# ==========================================================
if __name__ == "__main__":
    results, saturation = generate_plots_only()

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


PLOT GENERATION ONLY - No calculations (Titles REMOVED) - EPS FORMAT
Synthesizer: CTGAN
Randomization: RY_PR40
Report path: C:\Users\donatella.papa\OneDrive - ISTAT\WP_13_Istat_Eurostat\Step4\Output\Report\attack1_used
Plot output: C:\Users\donatella.papa\OneDrive - ISTAT\WP_13_Istat_Eurostat\Step4\Output\Plot
------------------------------------------------------------
Looking for results file: report_unificato_completo_RY_PR40_CTGAN.csv
Looking for saturation file: saturazione_modelli_RY_PR40_CTGAN.csv

Loading results from: report_unificato_completo_RY_PR40_CTGAN.csv
Loading saturation from: saturazione_modelli_RY_PR40_CTGAN.csv

Results DataFrame columns: ['modello', 'k', 'acc_real', 'acc_unseen', 'delta', 'fedelta', 'pct_tie_real', 'pct_tie_unseen']
Saturation DataFrame columns: ['modello', 'max_acc_real', 'k_max_real', 'max_acc_unseen', 'k_max_unseen', 'min_delta', 'sat_k_real', 'sat_k_unseen', 'tradeoff_k', 'tradeoff_acc', 'tradeoff_delta']

Loaded 507 result rows
Loaded 13 mode

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


  ✓ Accuracy plots (VR + BR) - no titles - EPS
  ✓ Delta analysis plot - no title - EPS

Saturation DataFrame columns: ['modello', 'max_acc_real', 'k_max_real', 'max_acc_unseen', 'k_max_unseen', 'min_delta', 'sat_k_real', 'sat_k_unseen', 'tradeoff_k', 'tradeoff_acc', 'tradeoff_delta']
Creating saturation column from data...


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


  ✓ Comparative heatmap - no title - EPS
------------------------------------------------------------
All EPS plots saved to: C:\Users\donatella.papa\OneDrive - ISTAT\WP_13_Istat_Eurostat\Step4\Output\Plot


In [22]:
# ==========================================================
# ONLY PLOTS - NO CALCULATIONS - NO TITLES - EPS + JPEG FORMAT
# Dynamic legend placement (inside plot, where space is)
# Removed: heatmap, delta plot
# EPS  → pubblicazione
# JPEG → anteprima rapida (150 DPI, stessa cartella)
#
# Legend is FIXED and IDENTICAL across all synthesizers:
# top-10 models ranked by average max_acc_real across all
# synthesizers, same order/color/marker everywhere.
# One shared legend file is saved once.
# ==========================================================

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path

# ==========================================================
# CONFIGURATION
# ==========================================================

class Config:
    R                  = 'Y'        # 'Y' = randomizzazione, 'N' = no
    PR                 = 40         # percentuale: 0, 10, 20, 30, 40, 100
    SYNTHESIZER_TYPE   = 'XGBoost_O0'    # CTGAN | TVAE | XGBoost_O0
    USERNAME           = 'donatella.papa'

    RANDOMIZATION_LABEL = f'RY_PR{PR}' if R == 'Y' else 'RN_PR0'

    BASE_PATH     = rf'G:\Users\{USERNAME}\OneDrive - ISTAT\WP_13_Istat_Eurostat'
    OUTPUT_REPORT = f'{BASE_PATH}\\Step4\\Output\\Report\\attack1_used'
    OUTPUT_PLOT   = f'{BASE_PATH}\\Step4\\Output\\Plot'

SYNTHESIZER_TYPES = ['CTGAN', 'TVAE', 'XGBoost_O0']

# ==========================================================
# COLOR PALETTE
# ==========================================================

COLOR_PALETTE = [
    '#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#6A994E',
    '#BC4A6C', '#1C7C54', '#D62828', '#003D5B', '#E09F3E',
    '#7B2CBF', '#F72585', '#4CC9F0',  # +3 colori extra
]

MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X', '*', '<', '>', 'H', 'd', 'p']  # 13 markers

LINESTYLES = ['-', '--', '-.', ':', (0, (3, 1, 1, 1)), (0, (5, 1)), (0, (1, 1))]  # più stili

# ==========================================================
# SHARED STYLE SETUP
# ==========================================================

def setup_plot_style():
    """White background, no titles, clean grid."""
    plt.rcParams.update({
        'figure.facecolor':    'white',
        'axes.facecolor':      'white',
        'savefig.facecolor':   'white',
        'font.size':           12,
        'axes.labelsize':      14,
        'axes.titlesize':      16,
        'xtick.labelsize':     12,
        'ytick.labelsize':     12,
        'legend.fontsize':     11,
        'axes.grid':           True,
        'grid.alpha':          0.3,
        'grid.linestyle':      '--',
        # EPS-safe fonts
        'ps.useafm':           True,
        'pdf.use14corefonts':  True,
        'text.usetex':         False,
    })


# ==========================================================
# BUILD FIXED ORDERING (shared across all synthesizers)
# ==========================================================

def build_fixed_model_order(all_saturation, model_col, top_n=13):
    """
    Concatenate saturation tables from all synthesizers, compute the
    mean max_acc_real per model across synthesizers, and return the
    top-N models sorted by that mean — descending.

    This guarantees a single, stable ordering used for every plot and
    for the one shared legend file.
    """
    combined = pd.concat(all_saturation, ignore_index=True)
    mean_acc = (combined
                .groupby(model_col)['max_acc_real']
                .mean()
                .sort_values(ascending=False))
    top_models  = mean_acc.head(top_n).index.tolist()
    model_labels = [m.replace('MM_', '').replace('GOWER', 'Gower')
                    for m in top_models]
    print(f'HM":{top_models}')
    return top_models, model_labels


# ==========================================================
# SAVE SHARED LEGEND (called once after order is fixed)
# ==========================================================

def save_shared_legend(top_models, model_labels, cfg, plot_path):
    """
    Build a dummy figure with one line per model (using the fixed
    colors/markers) purely to extract handles, then save the legend
    as a standalone EPS + JPEG file named without any synthesizer tag.
    """
    setup_plot_style()
    fig, ax = plt.subplots(figsize=(1, 1))

    for idx, label in enumerate(model_labels):
        ax.plot(
            [], [],
            marker          = MARKERS[idx % len(MARKERS)],
            linestyle       = LINESTYLES[idx % len(LINESTYLES)],
            color           = COLOR_PALETTE[idx],
            label           = label,
            linewidth       = 2,
            markersize      = 6,
            markeredgecolor = 'black',
            markeredgewidth = 0.8,
        )

    handles, labels = ax.get_legend_handles_labels()
    plt.close(fig)

    fig_leg = plt.figure(figsize=(14, 1.2))
    fig_leg.legend(
        handles, labels,
        loc='center',
        ncol=5,
        frameon=False,
        fontsize=12,
        handlelength=2.5,
        columnspacing=1.5,
    )
    plt.axis('off')

    legend_eps  = plot_path / f'legend_all_metrics_{cfg.RANDOMIZATION_LABEL}.eps'
    legend_jpeg = legend_eps.with_suffix('.jpeg')

    plt.savefig(legend_eps,  format='eps',  dpi=300,
                bbox_inches='tight', facecolor='white')
    plt.savefig(legend_jpeg, format='jpeg', dpi=150,
                bbox_inches='tight', facecolor='white')
    plt.close(fig_leg)

    print(f'  ✓ SHARED LEGEND EPS  → {legend_eps.name}')
    print(f'  ✓ SHARED LEGEND JPEG → {legend_jpeg.name}')


# ==========================================================
# DYNAMIC LEGEND PLACEMENT
# ==========================================================

def find_best_legend_loc(ax):
    """
    Returns the matplotlib loc-string that minimises overlap
    between the legend box and the plotted data.
    """
    xlim  = ax.get_xlim()
    ylim  = ax.get_ylim()
    xspan = xlim[1] - xlim[0]
    yspan = ylim[1] - ylim[0]

    all_pts = []
    for line in ax.get_lines():
        xd = np.asarray(line.get_xdata(), dtype=float)
        yd = np.asarray(line.get_ydata(), dtype=float)
        for x, y in zip(xd, yd):
            if xlim[0] <= x <= xlim[1] and ylim[0] <= y <= ylim[1]:
                all_pts.append(((x - xlim[0]) / xspan,
                                (y - ylim[0]) / yspan))

    fig = ax.get_figure()
    tmp_leg = ax.legend(loc='upper left')
    fig.canvas.draw()
    bb    = tmp_leg.get_window_extent()
    ax_bb = ax.get_window_extent()
    lw    = bb.width  / ax_bb.width
    lh    = bb.height / ax_bb.height
    tmp_leg.remove()

    loc_map = {
        'upper left':    (0,          1 - lh),
        'upper right':   (1 - lw,     1 - lh),
        'lower left':    (0,          0),
        'lower right':   (1 - lw,     0),
        'upper center':  (0.5 - lw/2, 1 - lh),
        'lower center':  (0.5 - lw/2, 0),
        'center left':   (0,          0.5 - lh/2),
        'center right':  (1 - lw,     0.5 - lh/2),
    }

    best_loc   = 'best'
    best_score = float('inf')

    for loc, (x0, y0) in loc_map.items():
        x1, y1 = x0 + lw, y0 + lh
        hits    = sum(1 for (px, py) in all_pts
                      if x0 <= px <= x1 and y0 <= py <= y1)
        penalty = 0.5 if loc in ('upper center', 'lower center',
                                  'center left',  'center right') else 0
        score = hits + penalty
        if score < best_score:
            best_score, best_loc = score, loc

    return best_loc


def apply_legend(ax):
    """Draw the legend in the best free spot inside the axes."""
    loc = find_best_legend_loc(ax)
    ax.legend(
        loc=loc,
        frameon=True,
        framealpha=0.92,
        edgecolor='lightgrey',
        fancybox=True,
        fontsize=11,
    )


# ==========================================================
# PLOT: VR and BR accuracy vs k  (fixed model order)
# ==========================================================

def plot_top_models_accuracy(results_df, cfg, plot_path,
                             top_models, model_labels):
    """
    Two separate EPS+JPEG plots (VR accuracy, BR accuracy).
    Uses the pre-computed fixed top_models / model_labels so that
    colors and markers are identical across all synthesizers.
    """
    setup_plot_style()

    results_model_col = 'modello' if 'modello' in results_df.columns else 'model'

    for metric, ylabel, suffix in [
        ('acc_real',   'TVR Accuracy', 'TVR'),
        ('acc_unseen', 'BVR Accuracy', 'BVR'),
    ]:
        fig, ax = plt.subplots(figsize=(11, 7))
        ax.set_facecolor('white')

        for idx, model in enumerate(top_models):
            df_m = (results_df[results_df[results_model_col] == model]
                    .sort_values('k'))
            ax.plot(
                df_m['k'],
                df_m[metric],
                marker          = MARKERS[idx % len(MARKERS)],
                linestyle       = LINESTYLES[idx % len(LINESTYLES)],
                color           = COLOR_PALETTE[idx],
                label           = model_labels[idx],
                linewidth       = 2,
                markersize      = 6,
                markeredgecolor = 'black',
                markeredgewidth = 0.8,
            )

        ax.set_xlabel(r'$k$', fontsize=14, fontweight='bold')
        ax.set_ylabel(ylabel, fontsize=14, fontweight='bold')
        ax.set_xlim(left=0)
        ax.set_ylim(bottom=0.25)
        ax.grid(True, alpha=0.3, linestyle='--')

        #apply_legend(ax)

        plt.tight_layout()
        out_eps  = (plot_path /
                    f'accuracy_vs_k_{suffix}_{cfg.SYNTHESIZER_TYPE}'
                    f'_{cfg.RANDOMIZATION_LABEL}.eps')
        out_jpeg = out_eps.with_suffix('.jpeg')

        plt.savefig(out_eps,  format='eps',  dpi=300,
                    bbox_inches='tight', facecolor='white')
        plt.savefig(out_jpeg, format='jpeg', dpi=150,
                    bbox_inches='tight', facecolor='white')
        plt.close()

        print(f'  ✓ EPS  → {out_eps.name}')
        print(f'  ✓ JPEG → {out_jpeg.name}')


# ==========================================================
# MAIN
# ==========================================================

def generate_plots_only():
    """
    1. Load saturation CSVs for all synthesizers to build the
       single fixed model ordering.
    2. Save one shared legend file.
    3. Loop over synthesizers and generate plots using that
       fixed ordering.
    """
    cfg = Config()

    report_path = Path(cfg.OUTPUT_REPORT)
    plot_path   = Path(cfg.OUTPUT_PLOT)
    plot_path.mkdir(parents=True, exist_ok=True)

    # ----------------------------------------------------------
    # PASS 1: load all saturation files to fix the model order
    # ----------------------------------------------------------
    all_saturation   = []
    loaded_data      = {}   # synth -> (results_df, saturation_df)
    model_col        = None

    print('=' * 60)
    print('PASS 1 — building fixed model ordering from all synthesizers')
    print('=' * 60)

    for synth in SYNTHESIZER_TYPES:
        results_file    = (report_path /
                           f'report_unificato_completo_{cfg.RANDOMIZATION_LABEL}'
                           f'_{synth}.csv')
        saturation_file = (report_path /
                           f'saturazione_modelli_{cfg.RANDOMIZATION_LABEL}'
                           f'_{synth}.csv')

        missing = False
        for f in (results_file, saturation_file):
            if not f.exists():
                print(f'  ERROR: file not found → {f}')
                missing = True

        if missing:
            print(f'  Skipping {synth} (missing files).\n')
            continue

        results_df    = pd.read_csv(results_file)
        saturation_df = pd.read_csv(saturation_file)

        mc = 'modello' if 'modello' in saturation_df.columns else 'model'
        if model_col is None:
            model_col = mc

        all_saturation.append(saturation_df)
        loaded_data[synth] = (results_df, saturation_df)
        print(f'  ✓ Loaded {synth}')

    if not all_saturation:
        print('No data loaded — aborting.')
        return {}

    # Build fixed order
    top_models, model_labels = build_fixed_model_order(
        all_saturation, model_col, top_n=10
    )
    print(f'\nFixed model order ({len(top_models)} models):')
    for i, (m, l) in enumerate(zip(top_models, model_labels)):
        print(f'  {i+1:2d}. {l}  ({m})')

    # Save the one shared legend
    print()
    save_shared_legend(top_models, model_labels, cfg, plot_path)

    # ----------------------------------------------------------
    # PASS 2: generate plots for each synthesizer
    # ----------------------------------------------------------
    print()
    print('=' * 60)
    print('PASS 2 — generating plots with fixed legend')
    print('=' * 60)

    for synth, (results_df, _) in loaded_data.items():
        cfg.SYNTHESIZER_TYPE = synth
        print(f'\n--- {synth} ---')
        plot_top_models_accuracy(
            results_df, cfg, plot_path, top_models, model_labels
        )

    print()
    print(f'All EPS + JPEG saved to: {plot_path}')
    print('=' * 60)

    return loaded_data


# ==========================================================
# ENTRY POINT
# ==========================================================

if __name__ == '__main__':
    all_results = generate_plots_only()

PASS 1 — building fixed model ordering from all synthesizers
  ✓ Loaded CTGAN
  ✓ Loaded TVAE
  ✓ Loaded XGBoost_O0
HM":['MM_canberra', 'MM_hamming', 'MM_cosine', 'MM_correlation', 'gower', 'MM_braycurtis', 'MM_cityblock', 'MM_manhattan', 'MM_l1', 'MM_sqeuclidean']

Fixed model order (10 models):
   1. canberra  (MM_canberra)
   2. hamming  (MM_hamming)
   3. cosine  (MM_cosine)
   4. correlation  (MM_correlation)
   5. gower  (gower)
   6. braycurtis  (MM_braycurtis)
   7. cityblock  (MM_cityblock)
   8. manhattan  (MM_manhattan)
   9. l1  (MM_l1)
  10. sqeuclidean  (MM_sqeuclidean)



The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


  ✓ SHARED LEGEND EPS  → legend_all_metrics_RY_PR40.eps
  ✓ SHARED LEGEND JPEG → legend_all_metrics_RY_PR40.jpeg

PASS 2 — generating plots with fixed legend

--- CTGAN ---


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


  ✓ EPS  → accuracy_vs_k_TVR_CTGAN_RY_PR40.eps
  ✓ JPEG → accuracy_vs_k_TVR_CTGAN_RY_PR40.jpeg


...

  ✓ EPS  → accuracy_vs_k_TVR_TVAE_RY_PR40.eps
  ✓ JPEG → accuracy_vs_k_TVR_TVAE_RY_PR40.jpeg


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


  ✓ EPS  → accuracy_vs_k_BVR_TVAE_RY_PR40.eps
  ✓ JPEG → accuracy_vs_k_BVR_TVAE_RY_PR40.jpeg

--- XGBoost_O0 ---


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


  ✓ EPS  → accuracy_vs_k_TVR_XGBoost_O0_RY_PR40.eps
  ✓ JPEG → accuracy_vs_k_TVR_XGBoost_O0_RY_PR40.jpeg
  ✓ EPS  → accuracy_vs_k_BVR_XGBoost_O0_RY_PR40.eps
  ✓ JPEG → accuracy_vs_k_BVR_XGBoost_O0_RY_PR40.jpeg

All EPS + JPEG saved to: G:\Users\donatella.papa\OneDrive - ISTAT\WP_13_Istat_Eurostat\Step4\Output\Plot


In [26]:
# ==========================================================
# ONLY PLOTS - NO CALCULATIONS - NO TITLES - EPS + JPEG FORMAT
# Dynamic legend placement (inside plot, where space is)
# Removed: heatmap, delta plot
# EPS  → pubblicazione
# JPEG → anteprima rapida (150 DPI, stessa cartella)
#
# Legend is FIXED and IDENTICAL across all synthesizers:
# top-13 models ranked by average max_acc_real across all
# synthesizers, same order/color/marker everywhere.
# One shared legend file is saved once.
# ==========================================================

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path

# ==========================================================
# CONFIGURATION
# ==========================================================

class Config:
    R                  = 'Y'        # 'Y' = randomizzazione, 'N' = no
    PR                 = 40         # percentuale: 0, 10, 20, 30, 40, 100
    SYNTHESIZER_TYPE   = 'XGBoost_O0'    # CTGAN | TVAE | XGBoost_O0
    USERNAME           = 'donatella.papa'

    RANDOMIZATION_LABEL = f'RY_PR{PR}' if R == 'Y' else 'RN_PR0'

    BASE_PATH     = rf'G:\Users\{USERNAME}\OneDrive - ISTAT\WP_13_Istat_Eurostat'
    OUTPUT_REPORT = f'{BASE_PATH}\\Step4\\Output\\Report\\attack1_used'
    OUTPUT_PLOT   = f'{BASE_PATH}\\Step4\\Output\\Plot'

SYNTHESIZER_TYPES = ['CTGAN', 'TVAE', 'XGBoost_O0']

# ==========================================================
# COLOR PALETTE (13 colors for 13 models)
# ==========================================================

COLOR_PALETTE = [
    '#2E86AB',  # Blue
    '#A23B72',  # Purple
    '#F18F01',  # Orange
    '#C73E1D',  # Red
    '#6A994E',  # Green
    '#BC4A6C',  # Pink
    '#1C7C54',  # Dark Green
    '#D62828',  # Bright Red
    '#003D5B',  # Navy
    '#E09F3E',  # Gold
    '#7B2CBF',  # Violet
    '#F72585',  # Hot Pink
    '#4CC9F0',  # Cyan
]

MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X', '*', '<', '>', 'H', 'd', 'p']

LINESTYLES = ['-', '--', '-.', ':', (0, (3, 1, 1, 1)), (0, (5, 1)), (0, (1, 1))]

# ==========================================================
# SHARED STYLE SETUP
# ==========================================================

def setup_plot_style():
    """White background, no titles, clean grid."""
    plt.rcParams.update({
        'figure.facecolor':    'white',
        'axes.facecolor':      'white',
        'savefig.facecolor':   'white',
        'font.size':           12,
        'axes.labelsize':      14,
        'axes.titlesize':      16,
        'xtick.labelsize':     12,
        'ytick.labelsize':     12,
        'legend.fontsize':     11,
        'axes.grid':           True,
        'grid.alpha':          0.3,
        'grid.linestyle':      '--',
        # EPS-safe fonts
        'ps.useafm':           True,
        'pdf.use14corefonts':  True,
        'text.usetex':         False,
    })


# ==========================================================
# BUILD FIXED ORDERING (shared across all synthesizers)
# Uses REPORTS (not saturation files) to get max_acc_real
# ==========================================================

def build_fixed_model_order_from_reports(all_reports, model_col, top_n=13):
    """
    Use complete reports (with data for each k) to compute
    max_acc_real per model across all synthesizers.
    """
    combined = pd.concat(all_reports, ignore_index=True)
    
    # Calculate max acc_real per model
    max_per_model = combined.groupby(model_col)['acc_real'].max()
    max_per_model = max_per_model.sort_values(ascending=False)
    
    if top_n is not None and top_n < len(max_per_model):
        top_models = max_per_model.head(top_n).index.tolist()
    else:
        top_models = max_per_model.index.tolist()
    
    # Clean model labels for legend
    model_labels = [m.replace('MM_', '').replace('GOWER_WEIGHTED', 'Gower')
                    for m in top_models]
    
    return top_models, model_labels


# ==========================================================
# SAVE SHARED LEGEND (called once after order is fixed)
# ==========================================================

def save_shared_legend(top_models, model_labels, cfg, plot_path):
    """
    Build a dummy figure with one line per model (using the fixed
    colors/markers) purely to extract handles, then save the legend
    as a standalone EPS + JPEG file named without any synthesizer tag.
    """
    setup_plot_style()
    fig, ax = plt.subplots(figsize=(1, 1))

    for idx, label in enumerate(model_labels):
        ax.plot(
            [], [],
            marker          = MARKERS[idx % len(MARKERS)],
            linestyle       = LINESTYLES[idx % len(LINESTYLES)],
            color           = COLOR_PALETTE[idx],
            label           = label,
            linewidth       = 2,
            markersize      = 6,
            markeredgecolor = 'black',
            markeredgewidth = 0.8,
        )

    handles, labels = ax.get_legend_handles_labels()
    plt.close(fig)

    # Calculate number of columns based on number of models
    n_models = len(model_labels)
    n_cols = min(7, n_models)  # max 7 columns, adjust as needed
    
    fig_leg = plt.figure(figsize=(14, 1.5))
    fig_leg.legend(
        handles, labels,
        loc='center',
        ncol=n_cols,
        frameon=False,
        fontsize=10,
        handlelength=2.0,
        columnspacing=1.2,
    )
    plt.axis('off')

    legend_eps  = plot_path / f'legend_all_metrics_{cfg.RANDOMIZATION_LABEL}.eps'
    legend_jpeg = legend_eps.with_suffix('.jpeg')

    plt.savefig(legend_eps,  format='eps',  dpi=300,
                bbox_inches='tight', facecolor='white')
    plt.savefig(legend_jpeg, format='jpeg', dpi=150,
                bbox_inches='tight', facecolor='white')
    plt.close(fig_leg)

    print(f'  ✓ SHARED LEGEND EPS  → {legend_eps.name}')
    print(f'  ✓ SHARED LEGEND JPEG → {legend_jpeg.name}')


# ==========================================================
# DYNAMIC LEGEND PLACEMENT
# ==========================================================

def find_best_legend_loc(ax):
    """
    Returns the matplotlib loc-string that minimises overlap
    between the legend box and the plotted data.
    """
    xlim  = ax.get_xlim()
    ylim  = ax.get_ylim()
    xspan = xlim[1] - xlim[0]
    yspan = ylim[1] - ylim[0]

    all_pts = []
    for line in ax.get_lines():
        xd = np.asarray(line.get_xdata(), dtype=float)
        yd = np.asarray(line.get_ydata(), dtype=float)
        for x, y in zip(xd, yd):
            if xlim[0] <= x <= xlim[1] and ylim[0] <= y <= ylim[1]:
                all_pts.append(((x - xlim[0]) / xspan,
                                (y - ylim[0]) / yspan))

    fig = ax.get_figure()
    tmp_leg = ax.legend(loc='upper left')
    fig.canvas.draw()
    bb    = tmp_leg.get_window_extent()
    ax_bb = ax.get_window_extent()
    lw    = bb.width  / ax_bb.width
    lh    = bb.height / ax_bb.height
    tmp_leg.remove()

    loc_map = {
        'upper left':    (0,          1 - lh),
        'upper right':   (1 - lw,     1 - lh),
        'lower left':    (0,          0),
        'lower right':   (1 - lw,     0),
        'upper center':  (0.5 - lw/2, 1 - lh),
        'lower center':  (0.5 - lw/2, 0),
        'center left':   (0,          0.5 - lh/2),
        'center right':  (1 - lw,     0.5 - lh/2),
    }

    best_loc   = 'best'
    best_score = float('inf')

    for loc, (x0, y0) in loc_map.items():
        x1, y1 = x0 + lw, y0 + lh
        hits    = sum(1 for (px, py) in all_pts
                      if x0 <= px <= x1 and y0 <= py <= y1)
        penalty = 0.5 if loc in ('upper center', 'lower center',
                                  'center left',  'center right') else 0
        score = hits + penalty
        if score < best_score:
            best_score, best_loc = score, loc

    return best_loc


def apply_legend(ax):
    """Draw the legend in the best free spot inside the axes."""
    loc = find_best_legend_loc(ax)
    ax.legend(
        loc=loc,
        frameon=True,
        framealpha=0.92,
        edgecolor='lightgrey',
        fancybox=True,
        fontsize=10,
    )


# ==========================================================
# PLOT: VR and BR accuracy vs k (fixed model order)
# ==========================================================

def plot_top_models_accuracy(results_df, cfg, plot_path,
                             top_models, model_labels):
    """
    Two separate EPS+JPEG plots (VR accuracy, BR accuracy).
    Uses the pre-computed fixed top_models / model_labels so that
    colors and markers are identical across all synthesizers.
    """
    setup_plot_style()

    results_model_col = 'modello' if 'modello' in results_df.columns else 'model'

    for metric, ylabel, suffix in [
    ('acc_real',   'TVR Accuracy', 'TVR'),
    ('acc_unseen', 'BVR Accuracy', 'BVR'),
]:
        fig, ax = plt.subplots(figsize=(11, 7))
        ax.set_facecolor('white')

        for idx, model in enumerate(top_models):
            # Check if model exists in this synthesizer's results
            if model not in results_df[results_model_col].values:
                print(f'  Warning: {model} not found in {cfg.SYNTHESIZER_TYPE}, skipping')
                continue
                
            df_m = (results_df[results_df[results_model_col] == model]
                    .sort_values('k'))
            
            if len(df_m) == 0:
                continue
                
            ax.plot(
                df_m['k'],
                df_m[metric],
                marker          = MARKERS[idx % len(MARKERS)],
                linestyle       = LINESTYLES[idx % len(LINESTYLES)],
                color           = COLOR_PALETTE[idx],
                label           = model_labels[idx],  # label ancora presente ma non usata
                linewidth       = 2,
                markersize      = 6,
                markeredgecolor = 'black',
                markeredgewidth = 0.8,
            )

        ax.set_xlabel(r'$k$', fontsize=14, fontweight='bold')
        ax.set_ylabel(ylabel, fontsize=14, fontweight='bold')
        ax.set_xlim(left=0)
        ax.set_ylim(bottom=0.25, top=0.55)
        ax.grid(True, alpha=0.3, linestyle='--')

        # apply_legend(ax)  <--- COMMENTATA/RIMOSSA

        plt.tight_layout()
        out_eps  = (plot_path /
                    f'accuracy_vs_k_{suffix}_{cfg.SYNTHESIZER_TYPE}'
                    f'_{cfg.RANDOMIZATION_LABEL}.eps')
        out_jpeg = out_eps.with_suffix('.jpeg')

        plt.savefig(out_eps,  format='eps',  dpi=300,
                    bbox_inches='tight', facecolor='white')
        plt.savefig(out_jpeg, format='jpeg', dpi=150,
                    bbox_inches='tight', facecolor='white')
        plt.close()

        print(f'  ✓ EPS  → {out_eps.name}')
        print(f'  ✓ JPEG → {out_jpeg.name}')


# ==========================================================
# MAIN
# ==========================================================

def generate_plots_only():
    """
    1. Load report CSVs for all synthesizers to build the
       single fixed model ordering (based on max_acc_real).
    2. Save one shared legend file.
    3. Loop over synthesizers and generate plots using that
       fixed ordering.
    """
    cfg = Config()

    report_path = Path(cfg.OUTPUT_REPORT)
    plot_path   = Path(cfg.OUTPUT_PLOT)
    plot_path.mkdir(parents=True, exist_ok=True)

    # ----------------------------------------------------------
    # PASS 1: load all REPORT files to fix the model order
    # ----------------------------------------------------------
    all_reports     = []
    loaded_data     = {}   # synth -> (results_df, saturation_df)
    model_col       = None

    print('=' * 60)
    print('PASS 1 — building fixed model ordering from REPORTS')
    print('=' * 60)

    for synth in SYNTHESIZER_TYPES:
        results_file    = (report_path /
                           f'report_unificato_completo_{cfg.RANDOMIZATION_LABEL}'
                           f'_{synth}.csv')
        saturation_file = (report_path /
                           f'saturazione_modelli_{cfg.RANDOMIZATION_LABEL}'
                           f'_{synth}.csv')

        missing = False
        for f in (results_file, saturation_file):
            if not f.exists():
                print(f'  ERROR: file not found → {f}')
                missing = True

        if missing:
            print(f'  Skipping {synth} (missing files).\n')
            continue

        results_df    = pd.read_csv(results_file)
        saturation_df = pd.read_csv(saturation_file)

        mc = 'modello' if 'modello' in results_df.columns else 'model'
        if model_col is None:
            model_col = mc

        all_reports.append(results_df)
        loaded_data[synth] = (results_df, saturation_df)
        print(f'  ✓ Loaded {synth}')

    if not all_reports:
        print('No data loaded — aborting.')
        return {}

    # Build fixed order using REPORTS (not saturation files)
    top_models, model_labels = build_fixed_model_order_from_reports(
        all_reports, model_col, top_n=13
    )
    
    print(f'\nFixed model order ({len(top_models)} models):')
    for i, (m, l) in enumerate(zip(top_models, model_labels)):
        print(f'  {i+1:2d}. {l}  ({m})')

    # Save the one shared legend
    print()
    save_shared_legend(top_models, model_labels, cfg, plot_path)

    # ----------------------------------------------------------
    # PASS 2: generate plots for each synthesizer
    # ----------------------------------------------------------
    print()
    print('=' * 60)
    print('PASS 2 — generating plots with fixed legend')
    print('=' * 60)

    for synth, (results_df, _) in loaded_data.items():
        cfg.SYNTHESIZER_TYPE = synth
        print(f'\n--- {synth} ---')
        plot_top_models_accuracy(
            results_df, cfg, plot_path, top_models, model_labels
        )

    print()
    print(f'All EPS + JPEG saved to: {plot_path}')
    print('=' * 60)

    return loaded_data


# ==========================================================
# ENTRY POINT
# ==========================================================

if __name__ == '__main__':
    all_results = generate_plots_only()

PASS 1 — building fixed model ordering from REPORTS
  ✓ Loaded CTGAN
  ✓ Loaded TVAE
  ✓ Loaded XGBoost_O0

Fixed model order (13 models):
   1. canberra  (MM_canberra)
   2. hamming  (MM_hamming)
   3. Gower  (Gower)
   4. cosine  (MM_cosine)
   5. correlation  (MM_correlation)
   6. braycurtis  (MM_braycurtis)
   7. l1  (MM_l1)
   8. cityblock  (MM_cityblock)
   9. manhattan  (MM_manhattan)
  10. sqeuclidean  (MM_sqeuclidean)
  11. l2  (MM_l2)
  12. euclidean  (MM_euclidean)
  13. nan_euclidean  (MM_nan_euclidean)



The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


  ✓ SHARED LEGEND EPS  → legend_all_metrics_RY_PR40.eps
  ✓ SHARED LEGEND JPEG → legend_all_metrics_RY_PR40.jpeg

PASS 2 — generating plots with fixed legend

--- CTGAN ---


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


  ✓ EPS  → accuracy_vs_k_TVR_CTGAN_RY_PR40.eps
  ✓ JPEG → accuracy_vs_k_TVR_CTGAN_RY_PR40.jpeg


...

  ✓ EPS  → accuracy_vs_k_TVR_TVAE_RY_PR40.eps
  ✓ JPEG → accuracy_vs_k_TVR_TVAE_RY_PR40.jpeg


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


  ✓ EPS  → accuracy_vs_k_BVR_TVAE_RY_PR40.eps
  ✓ JPEG → accuracy_vs_k_BVR_TVAE_RY_PR40.jpeg

--- XGBoost_O0 ---


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


  ✓ EPS  → accuracy_vs_k_TVR_XGBoost_O0_RY_PR40.eps
  ✓ JPEG → accuracy_vs_k_TVR_XGBoost_O0_RY_PR40.jpeg
  ✓ EPS  → accuracy_vs_k_BVR_XGBoost_O0_RY_PR40.eps
  ✓ JPEG → accuracy_vs_k_BVR_XGBoost_O0_RY_PR40.jpeg

All EPS + JPEG saved to: G:\Users\donatella.papa\OneDrive - ISTAT\WP_13_Istat_Eurostat\Step4\Output\Plot
